# День 09. Упражнение 03
# Ансамбли

## 0. Импорты

In [1]:
import pandas as pd  # (Пункт 1) Работа с таблицами и данными (DataFrame)
from sklearn.model_selection import train_test_split  # (Пункт 1) Разделение данных на train/valid/test
from sklearn.svm import SVC  # (Пункт 2, 3, 4, 5) Классификатор SVM для одиночных и ансамблевых моделей
from sklearn.tree import DecisionTreeClassifier  # (Пункт 2, 3, 5) Дерево решений для одиночных и ансамблевых моделей
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, BaggingClassifier, StackingClassifier
# (Пункт 2, 3, 4, 5)
# RandomForestClassifier — одиночная модель (2)
# VotingClassifier — голосующий ансамбль (3)
# BaggingClassifier — бэггинг-ансамбль (4)
# StackingClassifier — стекинг-ансамбль (5)

from sklearn.linear_model import LogisticRegression  # (Пункт 5) Финальный классификатор в стекинге
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix
# (Пункты 2–6) Метрики качества моделей и анализ ошибок

from sklearn.model_selection import StratifiedKFold  # (Пункт 5) Кросс-валидация для стекинга
from sklearn.multiclass import OneVsRestClassifier  # (Пункт 5) Обёртка для мультиклассовой логистической регрессии в стекинге
import joblib  # (Пункт 6) Сохранение обученной модели на диск
import numpy as np  # (Пункты 1, 6) Работа с массивами, анализ ошибок по классам

## 1. Предобработка

1. Создайте тот же DataFrame, что и в предыдущем упражнении.
2. Используйте `train_test_split` с параметрами `test_size=0.2`, `random_state=21`, чтобы получить `X_train`, `y_train`, `X_test`, `y_test`, а затем из предыдущих `X_train`, `y_train` получите `X_train`, `y_train`, `X_valid`, `y_valid`. Используйте дополнительный параметр `stratify`.

In [3]:
# Загружаем основной датасет
df = pd.read_csv('../data/day-of-week-not-scaled.csv')
df_day = pd.read_csv('../data/dayofweek.csv')
df['dayofweek'] = df_day['dayofweek']

X = df.drop('dayofweek', axis=1)
y = df['dayofweek']

In [4]:
# Сначала train/test split
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)

In [5]:
# Затем train/valid split из train
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=21, stratify=y_train_full
)

## 2. Индивидуальные классификаторы

1. Обучите SVM, дерево решений и случайный лес с лучшими параметрами из упражнения 01, для всех используйте `random_state=21`.
2. Оцените `accuracy`, `precision` и `recall` на валидационной выборке.
3. Результат каждой ячейки должен выглядеть так:

```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
```

In [6]:
# SVM с лучшими параметрами
svm = SVC(
    kernel='linear',      
    C=1,                  
    gamma='scale',        
    class_weight=None,    
    probability=True,
    random_state=21
)
svm.fit(X_train, y_train)
y_pred = svm.predict(X_valid)

print(f"SVM:")
print(f"accuracy is {accuracy_score(y_valid, y_pred):.5f}")
print(f"precision is {precision_score(y_valid, y_pred, average='weighted'):.5f}")
print(f"recall is {recall_score(y_valid, y_pred, average='weighted'):.5f}")

SVM:
accuracy is 0.60741
precision is 0.60406
recall is 0.60741


In [7]:
# Decision Tree
tree = DecisionTreeClassifier(
    max_depth=8,
    class_weight=None,
    criterion='entropy',
    random_state=21
)
tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_valid)
print(f"Decision Tree:")
print(f"accuracy is {accuracy_score(y_valid, y_pred_tree):.5f}")
print(f"precision is {precision_score(y_valid, y_pred_tree, average='weighted'):.5f}")
print(f"recall is {recall_score(y_valid, y_pred_tree, average='weighted'):.5f}")
print()

Decision Tree:
accuracy is 0.67407
precision is 0.70552
recall is 0.67407



In [8]:
# Random Forest
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    class_weight=None,
    criterion='entropy',
    random_state=21
)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_valid)
print(f"Random Forest:")
print(f"accuracy is {accuracy_score(y_valid, y_pred_rf):.5f}")
print(f"precision is {precision_score(y_valid, y_pred_rf, average='weighted'):.5f}")
print(f"recall is {recall_score(y_valid, y_pred_rf, average='weighted'):.5f}")

Random Forest:
accuracy is 0.77037
precision is 0.80172
recall is 0.77037


## 3. Голосующие классификаторы


1. Используя `VotingClassifier` и три обученные модели, вычислите `accuracy`, `precision` и `recall` на валидационной выборке.
2. Поиграйте с другими параметрами.
3. Вычислите `accuracy`, `precision` и `recall` на тестовой выборке для модели с лучшими весами по точности (если несколько моделей с одинаковой точностью — выберите ту, у которой precision выше).


In [9]:
# VotingClassifier: soft voting (все модели поддерживают predict_proba)
voting_soft = VotingClassifier(
    estimators=[
        ('svm', svm),
        ('tree', tree),
        ('rf', rf)
    ],
    voting='soft'
)
voting_soft.fit(X_train, y_train)
y_pred_soft = voting_soft.predict(X_valid)

print("VotingClassifier (soft):")
print(f"accuracy is {accuracy_score(y_valid, y_pred_soft):.5f}")
print(f"precision is {precision_score(y_valid, y_pred_soft, average='weighted'):.5f}")
print(f"recall is {recall_score(y_valid, y_pred_soft, average='weighted'):.5f}")

VotingClassifier (soft):
accuracy is 0.73704
precision is 0.75781
recall is 0.73704


In [10]:
# VotingClassifier: hard voting
voting_hard = VotingClassifier(
    estimators=[
        ('svm', svm),
        ('tree', tree),
        ('rf', rf)
    ],
    voting='hard'
)
voting_hard.fit(X_train, y_train)
y_pred_hard = voting_hard.predict(X_valid)

print("VotingClassifier (hard):")
print(f"accuracy is {accuracy_score(y_valid, y_pred_hard):.5f}")
print(f"precision is {precision_score(y_valid, y_pred_hard, average='weighted'):.5f}")
print(f"recall is {recall_score(y_valid, y_pred_hard, average='weighted'):.5f}")

VotingClassifier (hard):
accuracy is 0.73704
precision is 0.75492
recall is 0.73704


In [11]:
# Выбор лучшей модели по accuracy (если равны — по precision)
results = [
    {
        'name': 'soft',
        'accuracy': accuracy_score(y_valid, y_pred_soft),
        'precision': precision_score(y_valid, y_pred_soft, average='weighted'),
        'recall': recall_score(y_valid, y_pred_soft, average='weighted'),
        'model': voting_soft
    },
    {
        'name': 'hard',
        'accuracy': accuracy_score(y_valid, y_pred_hard),
        'precision': precision_score(y_valid, y_pred_hard, average='weighted'),
        'recall': recall_score(y_valid, y_pred_hard, average='weighted'),
        'model': voting_hard
    }
]
# Сортируем по accuracy, затем по precision
best = sorted(results, key=lambda x: (x['accuracy'], x['precision']), reverse=True)[0]
print(f"Лучшая модель: VotingClassifier ({best['name']})")

Лучшая модель: VotingClassifier (soft)


In [12]:
# Метрики для лучшей модели на тестовой выборке
best['model'].fit(X_train_full, y_train_full)
y_pred_test = best['model'].predict(X_test)
print(f"VotingClassifier ({best['name']}) на тестовой выборке:")
print(f"accuracy is {accuracy_score(y_test, y_pred_test):.5f}")
print(f"precision is {precision_score(y_test, y_pred_test, average='weighted'):.5f}")
print(f"recall is {recall_score(y_test, y_pred_test, average='weighted'):.5f}")

VotingClassifier (soft) на тестовой выборке:
accuracy is 0.77219
precision is 0.79897
recall is 0.77219


**Комментарий:**  
VotingClassifier — это ансамблевый классификатор из библиотеки scikit-learn, который объединяет несколько моделей (например, SVM, дерево решений, случайный лес) и принимает итоговое решение путём голосования.

1. Как работает VotingClassifier:
    - voting='hard' — "жёсткое" голосование: каждый классификатор отдаёт свой голос за класс, и побеждает класс с большинством голосов.
    - voting='soft' — "мягкое" голосование: используются вероятности классов (predict_proba), и итоговый класс выбирается по максимальной суммарной вероятности.

2. Что такое "веса" (weights)  
Веса (weights) — это параметр VotingClassifier, который позволяет "усилить" вклад отдельных моделей в итоговое голосование.
Например, если одна модель работает лучше других, ей можно дать больший вес:

## 4. Бэггинг-классификаторы


1. Используя `BaggingClassifier` и SVM с лучшими параметрами, создайте ансамбль, попробуйте разные значения `n_estimators`, используйте `random_state=21`.
2. Поиграйте с другими параметрами.
3. Вычислите `accuracy`, `precision` и `recall` для модели с лучшими параметрами (по точности) на тестовой выборке (если несколько моделей с одинаковой точностью — выберите ту, у которой precision выше).


In [13]:
# 1. BaggingClassifier с SVM и разными n_estimators (scikit-learn >= 1.2)
for n in [5, 10, 25, 50]:
    bagging = BaggingClassifier(
        estimator=SVC(
            kernel='linear',
            C=1,
            gamma='scale',
            class_weight=None,
            probability=True,
            random_state=21
        ),
        n_estimators=n,
        random_state=21,
        n_jobs=-1
    )
    bagging.fit(X_train, y_train)
    y_pred_bag = bagging.predict(X_valid)
    print(f"BaggingClassifier (n_estimators={n}):")
    print(f"accuracy is {accuracy_score(y_valid, y_pred_bag):.5f}")
    print(f"precision is {precision_score(y_valid, y_pred_bag, average='weighted'):.5f}")
    print(f"recall is {recall_score(y_valid, y_pred_bag, average='weighted'):.5f}")
    print()

BaggingClassifier (n_estimators=5):
accuracy is 0.64074
precision is 0.63780
recall is 0.64074

BaggingClassifier (n_estimators=10):
accuracy is 0.63333
precision is 0.63073
recall is 0.63333

BaggingClassifier (n_estimators=25):
accuracy is 0.67037
precision is 0.67981
recall is 0.67037

BaggingClassifier (n_estimators=50):
accuracy is 0.66667
precision is 0.67406
recall is 0.66667



In [16]:
# 2. BaggingClassifier: играем с параметрами max_samples и max_features
for max_samples in [0.5, 0.7, 1.0]:
    for max_features in [0.5, 0.7, 1.0]:
        bagging = BaggingClassifier(
            estimator=SVC(
                kernel='rbf',
                C=10,
                gamma='auto',
                probability=True,
                random_state=21
            ),
            n_estimators=50,  # ← важно
            max_samples=max_samples,
            max_features=max_features,
            random_state=21,
            n_jobs=-1
        )
        bagging.fit(X_train, y_train)
        y_pred_bag = bagging.predict(X_valid)
        print(f"Bagging (n_estimators=25, max_samples={max_samples}, max_features={max_features}):")
        print(f"accuracy is {accuracy_score(y_valid, y_pred_bag):.5f}")
        print(f"precision is {precision_score(y_valid, y_pred_bag, average='weighted'):.5f}")
        print(f"recall is {recall_score(y_valid, y_pred_bag, average='weighted'):.5f}")
        print()

Bagging (n_estimators=25, max_samples=0.5, max_features=0.5):
accuracy is 0.65185
precision is 0.74547
recall is 0.65185

Bagging (n_estimators=25, max_samples=0.5, max_features=0.7):
accuracy is 0.74444
precision is 0.78608
recall is 0.74444

Bagging (n_estimators=25, max_samples=0.5, max_features=1.0):
accuracy is 0.83333
precision is 0.84955
recall is 0.83333

Bagging (n_estimators=25, max_samples=0.7, max_features=0.5):
accuracy is 0.68519
precision is 0.75271
recall is 0.68519

Bagging (n_estimators=25, max_samples=0.7, max_features=0.7):
accuracy is 0.78889
precision is 0.80921
recall is 0.78889

Bagging (n_estimators=25, max_samples=0.7, max_features=1.0):
accuracy is 0.85556
precision is 0.86899
recall is 0.85556

Bagging (n_estimators=25, max_samples=1.0, max_features=0.5):
accuracy is 0.74815
precision is 0.78443
recall is 0.74815

Bagging (n_estimators=25, max_samples=1.0, max_features=0.7):
accuracy is 0.84444
precision is 0.85762
recall is 0.84444

Bagging (n_estimators=25

In [17]:
# 3. Выбор лучшей модели по accuracy (если равны — по precision) и метрики на тесте
bagging_results = []
for n in [5, 10, 25, 50]:
    bagging = BaggingClassifier(
        estimator=SVC(
            kernel='linear',
            C=1,
            gamma='scale',
            class_weight=None,
            probability=True,
            random_state=21
        ),
        n_estimators=n,
        random_state=21,
        n_jobs=-1
    )
    bagging.fit(X_train, y_train)
    y_pred_bag = bagging.predict(X_valid)
    bagging_results.append({
        'n_estimators': n,
        'accuracy': accuracy_score(y_valid, y_pred_bag),
        'precision': precision_score(y_valid, y_pred_bag, average='weighted'),
        'recall': recall_score(y_valid, y_pred_bag, average='weighted'),
        'model': bagging
    })

In [18]:
# Сортируем по accuracy, затем по precision
best_bagging = sorted(bagging_results, key=lambda x: (x['accuracy'], x['precision']), reverse=True)[0]
print(f"Лучшая модель: BaggingClassifier (n_estimators={best_bagging['n_estimators']})")
print(f"accuracy is {best_bagging['accuracy']:.5f}")
print(f"precision is {best_bagging['precision']:.5f}")
print(f"recall is {best_bagging['recall']:.5f}")

Лучшая модель: BaggingClassifier (n_estimators=25)
accuracy is 0.67037
precision is 0.67981
recall is 0.67037


In [19]:
# Метрики на тестовой выборке
best_bagging['model'].fit(X_train_full, y_train_full)
y_pred_bag_test = best_bagging['model'].predict(X_test)
print(f"BaggingClassifier (n_estimators={best_bagging['n_estimators']}) на тестовой выборке:")
print(f"accuracy is {accuracy_score(y_test, y_pred_bag_test):.5f}")
print(f"precision is {precision_score(y_test, y_pred_bag_test, average='weighted'):.5f}")
print(f"recall is {recall_score(y_test, y_pred_bag_test, average='weighted'):.5f}")

BaggingClassifier (n_estimators=25) на тестовой выборке:
accuracy is 0.70118
precision is 0.71562
recall is 0.70118


## 5. Стекинг-классификаторы


1. Для воспроизводимости создайте объект генератора кросс-валидации: `StratifiedKFold(n_splits=n, shuffle=True, random_state=21)`, где `n` нужно подобрать (см. ниже).
2. Используя `StackingClassifier` и три обученные модели, вычислите `accuracy`, `precision` и `recall` на валидационной выборке, попробуйте разные значения `n_splits` `[2, 3, 4, 5, 6, 7]` в генераторе кросс-валидации и параметр `passthrough` в самом классификаторе.
3. Вычислите `accuracy`, `precision` и `recall` для модели с лучшими параметрами (по точности) на тестовой выборке (если несколько моделей с одинаковой точностью — выберите ту, у которой precision выше). Используйте `final_estimator=LogisticRegression(solver='liblinear')`.


In [20]:
# 1. Для воспроизводимости создаём генератор кросс-валидации
# Пример для n_splits=5 
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=21)
print(cv)

StratifiedKFold(n_splits=5, random_state=21, shuffle=True)


In [21]:
# 2. StackingClassifier: эксперименты с n_splits и passthrough
stacking_results = []
for n_splits in [2, 3, 4, 5, 6, 7]:
    for passthrough in [False, True]:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=21)
        stacking = StackingClassifier(
            estimators=[
                ('svm', svm),
                ('tree', tree),
                ('rf', rf)
            ],
            final_estimator=OneVsRestClassifier(LogisticRegression(solver='liblinear')),
            cv=cv,
            passthrough=passthrough,
            n_jobs=-1
        )
        stacking.fit(X_train, y_train)
        y_pred_stack = stacking.predict(X_valid)
        stacking_results.append({
            'n_splits': n_splits,
            'passthrough': passthrough,
            'accuracy': accuracy_score(y_valid, y_pred_stack),
            'precision': precision_score(y_valid, y_pred_stack, average='weighted'),
            'recall': recall_score(y_valid, y_pred_stack, average='weighted'),
            'model': stacking
        })
        print(f"StackingClassifier (n_splits={n_splits}, passthrough={passthrough}):")
        print(f"accuracy is {accuracy_score(y_valid, y_pred_stack):.5f}")
        print(f"precision is {precision_score(y_valid, y_pred_stack, average='weighted'):.5f}")
        print(f"recall is {recall_score(y_valid, y_pred_stack, average='weighted'):.5f}")
        print()

StackingClassifier (n_splits=2, passthrough=False):
accuracy is 0.80000
precision is 0.80823
recall is 0.80000

StackingClassifier (n_splits=2, passthrough=True):
accuracy is 0.79630
precision is 0.79833
recall is 0.79630

StackingClassifier (n_splits=3, passthrough=False):
accuracy is 0.79259
precision is 0.80278
recall is 0.79259

StackingClassifier (n_splits=3, passthrough=True):
accuracy is 0.79259
precision is 0.79502
recall is 0.79259

StackingClassifier (n_splits=4, passthrough=False):
accuracy is 0.79259
precision is 0.80006
recall is 0.79259

StackingClassifier (n_splits=4, passthrough=True):
accuracy is 0.80000
precision is 0.79970
recall is 0.80000

StackingClassifier (n_splits=5, passthrough=False):
accuracy is 0.80370
precision is 0.81229
recall is 0.80370

StackingClassifier (n_splits=5, passthrough=True):
accuracy is 0.80741
precision is 0.80939
recall is 0.80741

StackingClassifier (n_splits=6, passthrough=False):
accuracy is 0.80370
precision is 0.81124
recall is 0.803

In [22]:
# 3. Выбор лучшей stacking-модели и метрики на тестовой выборке
best_stack = sorted(stacking_results, key=lambda x: (x['accuracy'], x['precision']), reverse=True)[0]
print(f"Лучшая модель: StackingClassifier (n_splits={best_stack['n_splits']}, passthrough={best_stack['passthrough']})")
print(f"accuracy is {best_stack['accuracy']:.5f}")
print(f"precision is {best_stack['precision']:.5f}")
print(f"recall is {best_stack['recall']:.5f}")

Лучшая модель: StackingClassifier (n_splits=6, passthrough=True)
accuracy is 0.81111
precision is 0.81294
recall is 0.81111


In [23]:
# Оценка на тестовой выборке
best_stack['model'].fit(X_train_full, y_train_full)
y_pred_stack_test = best_stack['model'].predict(X_test)
print(f"StackingClassifier (n_splits={best_stack['n_splits']}, passthrough={best_stack['passthrough']}) на тестовой выборке:")
print(f"accuracy is {accuracy_score(y_test, y_pred_stack_test):.5f}")
print(f"precision is {precision_score(y_test, y_pred_stack_test, average='weighted'):.5f}")
print(f"recall is {recall_score(y_test, y_pred_stack_test, average='weighted'):.5f}")

StackingClassifier (n_splits=6, passthrough=True) на тестовой выборке:
accuracy is 0.83728
precision is 0.84954
recall is 0.83728


## 6. Предсказания

1. Выберите лучшую модель по точности (если несколько моделей с одинаковой точностью — выберите ту, у которой precision выше).
2. Проанализируйте: для какого дня недели ваша модель делает больше всего ошибок (в % от общего числа примеров этого класса в полном датасете), для какого labname и для каких users.
3. Сохраните модель.

In [24]:
# 1. Лучшая модель (например, stacking)
final_model = best_stack['model']
final_model.fit(X_train_full, y_train_full)
y_pred_final = final_model.predict(X_test)

In [25]:
# 2. Анализ ошибок по дню недели
cm = confusion_matrix(y_test, y_pred_final, labels=np.unique(y_test))
errors = (cm.sum(axis=1) - np.diag(cm)) / cm.sum(axis=1)
for i, day in enumerate(np.unique(y_test)):
    print(f'Доля ошибок для дня недели {day}: {errors[i]:.2%}')

Доля ошибок для дня недели 0: 40.74%
Доля ошибок для дня недели 1: 10.91%
Доля ошибок для дня недели 2: 30.00%
Доля ошибок для дня недели 3: 5.00%
Доля ошибок для дня недели 4: 38.10%
Доля ошибок для дня недели 5: 12.96%
Доля ошибок для дня недели 6: 14.08%


In [26]:
# 3. Анализ ошибок по labname
lab_cols = [col for col in X_test.columns if col.startswith('labname_')]
for lab in lab_cols:
    idx = X_test[lab] == 1
    if idx.sum() == 0:
        continue
    err = (y_test[idx] != y_pred_final[idx]).sum() / idx.sum()
    print(f'Доля ошибок для {lab}: {err:.2%}')

Доля ошибок для labname_code_rvw: 15.38%
Доля ошибок для labname_lab03: 100.00%
Доля ошибок для labname_lab03s: 100.00%
Доля ошибок для labname_lab05s: 16.67%
Доля ошибок для labname_laba04: 28.57%
Доля ошибок для labname_laba04s: 24.00%
Доля ошибок для labname_laba05: 4.26%
Доля ошибок для labname_laba06: 11.11%
Доля ошибок для labname_laba06s: 26.67%
Доля ошибок для labname_project1: 14.52%


In [27]:
# 4. Анализ ошибок по users
user_cols = [col for col in X_test.columns if col.startswith('uid_user_')]
for user in user_cols:
    idx = X_test[user] == 1
    if idx.sum() == 0:
        continue
    err = (y_test[idx] != y_pred_final[idx]).sum() / idx.sum()
    print(f'Доля ошибок для {user}: {err:.2%}')

Доля ошибок для uid_user_1: 0.00%
Доля ошибок для uid_user_10: 16.67%
Доля ошибок для uid_user_12: 0.00%
Доля ошибок для uid_user_13: 17.65%
Доля ошибок для uid_user_14: 12.90%
Доля ошибок для uid_user_15: 0.00%
Доля ошибок для uid_user_16: 40.00%
Доля ошибок для uid_user_17: 28.57%
Доля ошибок для uid_user_18: 50.00%
Доля ошибок для uid_user_19: 26.32%
Доля ошибок для uid_user_2: 7.14%
Доля ошибок для uid_user_20: 10.00%
Доля ошибок для uid_user_21: 21.43%
Доля ошибок для uid_user_22: 0.00%
Доля ошибок для uid_user_23: 0.00%
Доля ошибок для uid_user_24: 18.18%
Доля ошибок для uid_user_25: 27.27%
Доля ошибок для uid_user_26: 5.88%
Доля ошибок для uid_user_27: 16.67%
Доля ошибок для uid_user_28: 12.50%
Доля ошибок для uid_user_29: 18.18%
Доля ошибок для uid_user_3: 14.29%
Доля ошибок для uid_user_30: 37.50%
Доля ошибок для uid_user_31: 11.11%
Доля ошибок для uid_user_4: 22.22%
Доля ошибок для uid_user_6: 25.00%
Доля ошибок для uid_user_8: 0.00%


In [28]:
# 5. Сохраняем модель
joblib.dump(final_model, '../data/best_ensemble_model.joblib')

['../data/best_ensemble_model.joblib']